# AI/ML Football Analysis Project

## Project Overview
This project builds an end-to-end AI/ML Football Analysis system. We will use computer vision techniques to track players, referees, and the ball, assign players to teams based on jersey colors, and estimate their speed and distance covered.

### Problem Statement
Analyzing football match footage is challenging due to camera movement, occlusions, and the fast pace of the game. Our goal is to extract meaningful analytics (possession, speed, distance) from raw broadcast video using deep learning.

### Dataset
- **Kaggle Dataset**: DFL Bundesliga Data Shootout (for raw video clips)
- **Roboflow Dataset**: Custom annotated football dataset used for fine-tuning YOLOv8.

## 1. Setup & Installation

In [ ]:
!pip install ultralytics opencv-python scikit-learn pandas numpy matplotlib filterpy lapx

## 2. Import Libraries

In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import matplotlib.pyplot as plt
import sys
import os

# Import custom modules (ensure you are running this in the project root)
from trackers import Tracker
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator
from utils import read_video, save_video

## 3. Data Collection & Preprocessing
We load a sample video clip to perform EDA on the frames.

In [ ]:
video_path = 'input_videos/sample_match.mp4' # Replace with actual path
# Read video frames
try:
    video_frames = read_video(video_path)
    print(f"Loaded {len(video_frames)} frames from {video_path}")
except:
    print("Please place a video file in input_videos/ and update the path above.")

## 4. Model Building (YOLOv8)
We load a pre-trained YOLOv8 model. In a full training pipeline, we would train it using `model.train(data='data.yaml', epochs=100)`. Here we load the trained weights `best.pt` or use the standard `yolov8x.pt`.

In [ ]:
model_path = 'models/best.pt' if os.path.exists('models/best.pt') else 'yolov8x.pt'
tracker = Tracker(model_path)
print(f"Loaded model from {model_path}")

## 5. Evaluation & Inference
We run the tracker on our video frames.

In [ ]:
# Get object tracks
if 'video_frames' in locals():
    tracks = tracker.get_object_tracks(video_frames, read_from_stub=False)
    tracker.add_position_to_tracks(tracks)
    print("Tracking complete.")

## 6. Advanced Analytics (Optical Flow & K-Means)
We apply Optical Flow for camera movement, View Transformation for perspective, and K-Means clustering for team color assignment.

In [ ]:
if 'video_frames' in locals():
    # Camera movement estimator
    camera_estimator = CameraMovementEstimator(video_frames[0])
    camera_movement_per_frame = camera_estimator.get_camera_movement(video_frames, read_from_stub=False)
    camera_estimator.add_adjust_positions_to_tracks(tracks, camera_movement_per_frame)

    # View Transformer
    view_transformer = ViewTransformer()
    view_transformer.add_transformed_position_to_tracks(tracks)

    # Interpolate Ball Positions
    tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"])

    # Speed and distance estimator
    speed_and_distance_estimator = SpeedAndDistance_Estimator()
    speed_and_distance_estimator.add_speed_and_distance_to_tracks(tracks)

    # Assign Player Teams
    team_assigner = TeamAssigner()
    team_assigner.assign_team_color(video_frames[0], tracks['players'][0])
    
    for frame_num, player_track in enumerate(tracks['players']):
        for player_id, track in player_track.items():
            team = team_assigner.get_player_team(video_frames[frame_num], track['bbox'], player_id)
            tracks['players'][frame_num][player_id]['team'] = team 
            tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors[team]

    # Assign Ball Aquisition
    player_assigner = PlayerBallAssigner()
    team_ball_control = []
    for frame_num, player_track in enumerate(tracks['players']):
        ball_bbox = tracks['ball'][frame_num][1]['bbox']
        assigned_player = player_assigner.assign_ball_to_player(player_track, ball_bbox)

        if assigned_player != -1:
            tracks['players'][frame_num][assigned_player]['has_ball'] = True
            team_ball_control.append(tracks['players'][frame_num][assigned_player]['team'])
        else:
            if team_ball_control:
                team_ball_control.append(team_ball_control[-1])
            else:
                team_ball_control.append(1) 
    
    team_ball_control = np.array(team_ball_control)
    print("Advanced analytics complete.")

## 7. Saving the Processed Video
Draw all annotations (bounding boxes, team colors, speed, distance) and save the video.

In [ ]:
if 'video_frames' in locals():
    output_video_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control)
    output_video_frames = camera_estimator.draw_camera_movement(output_video_frames, camera_movement_per_frame)
    speed_and_distance_estimator.draw_speed_and_distance(output_video_frames, tracks)

    output_path = "output_videos/analysis_result.avi"
    os.makedirs("output_videos", exist_ok=True)
    save_video(output_video_frames, output_path)
    print(f"Processed video saved to {output_path}")

## 8. Deployment Links

- **GitHub Repository**: [Your GitHub Repo Link Here]
- **Streamlit Application**: [Your Streamlit App URL Here]